In [ ]:
!pip install spatialdata

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import random
import spatialdata as sd
from pathlib import Path

from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

In [3]:
import scanpy as sc
adata = sc.read_h5ad("G:/My Drive/Thesis/Projects/Data/Breast_Cancer/ann_data.h5ad")

In [ ]:
mask = adata.obs['annotation'] == 'DCIS_2'
adata.obs['cell_id'][mask][:70]

In [ ]:
mask = adata.obs['annotation'] == 'Invasive'
adata.obs['cell_id'][mask][:70]

In [2]:
#data_path = "G:/My Drive/Thesis/Projects/Data/Breast_Cancer/ann_data.zarr"
data_path = "G:/My Drive/Thesis/Projects/Data/Mouse_Brain_Coronal/data.zarr"
#data_path = "/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain_Coronal/data.zarr"
sdata = sd.read_zarr(data_path)
sdata

C:\Users\User\AppData\Local\Temp\ipykernel_28020\2708225495.py:4: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = sd.read_zarr(data_path)
no parent found for <ome_zarr.reader.Label object at 0x00000218EBFBB4D0>: None
no parent found for <ome_zarr.reader.Label object at 0x00000218EAD8C050>: None
c:\Users\User\Desktop\Xenium Data\Xenium_venv\Lib\site-packages\zarr\core\group.py:3535: ZarrUserWarning: Object at zmetadata is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


SpatialData object, with associated Zarr store: G:\My Drive\Thesis\Projects\Data\Mouse_Brain_Coronal\data.zarr
├── Images
│     ├── 'he_image': DataTree[cyx] (3, 24689, 17051), (3, 12344, 8525), (3, 6172, 4262), (3, 3086, 2131), (3, 1543, 1065)
│     └── 'morphology_focus': DataTree[cyx] (4, 23912, 34154), (4, 11956, 17077), (4, 5978, 8538), (4, 2989, 4269), (4, 1494, 2134)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (23912, 34154), (11956, 17077), (5978, 8538), (2989, 4269), (1494, 2134)
│     └── 'nucleus_labels': DataTree[yx] (23912, 34154), (11956, 17077), (5978, 8538), (2989, 4269), (1494, 2134)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 13) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (63173, 1) (2D shapes)
│     ├── 'cell_circles': GeoDataFrame shape: (63173, 2) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (63036, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (63173, 5006)
with coordinate system

In [3]:
from spatialdata import get_extent

img = sdata.images['he_image']
# Get the cropped H&E image (already aligned)
he = sd.transform(img, to_coordinate_system="global")
he_m = he['scale0'].image.data.compute()
extent = get_extent(he)

y_s,y_e = extent['y']
x_s,x_e = extent['x']

print('X_start: ', x_s)
print('y_start: ', y_s)

X_start:  1569.685642189309
y_start:  887.5043173028462


In [4]:
import warnings
warnings.filterwarnings('ignore')


# Transform the cell boundaries to global coordinates
navi = sd.transform(
    sdata.shapes['cell_boundaries'],
    to_coordinate_system="global"
)

path = "C:/Users/User/Desktop/Xenium Data/Mouse_Brain/Croped_Images_3"
os.makedirs(path)
l = os.listdir(path)

for ind in tqdm(sdata.tables['table'].obs.index, dynamic_ncols=True):

    id = sdata.tables['table'].obs['cell_id'][int(ind)]
    if f"{id}.png" in l:
      continue
    else:

      try:

        cell_center = navi.centroid[id]

        # Get cell center (yx format for plotting)
        x, y = cell_center.x, cell_center.y

        # Define crop size (e.g. 56 microns around the cell)
        #half_size = 70/0.363788  # adjust depending on how much context you want
        half_size = 70/0.2737
        xmin, xmax = x - half_size, x + half_size
        ymin, ymax = y - half_size, y + half_size

        try:
          c_img = he_m[:, int(ymin-y_s):int(ymax-y_s), int(xmin-x_s):int(xmax-x_s)]
        except Exception as e:
          print('error: ',e)
          c_img = None

        if c_img is not None and c_img.ndim >= 3 and c_img.shape[1] > 300 and c_img.shape[2] > 300:
          ## Convert to numpy
          he_crop_array = c_img
          normalized_he = he_crop_array.transpose(1, 2, 0)  # (cyx) → (yx, x, c)

        else:
          print(f'id: {id}', type(c_img), 'center: ', cell_center)
          normalized_he = np.zeros_like(normalized_he)

        #plt.imsave(f"/content/Croped_Images/{id}.png", normalized_he)
        plt.imsave(os.path.join(path, f"{id}.png"), normalized_he)

      except Exception as e:
        print(e)
        print(f'id: {id}', type(c_img), 'center: ', cell_center)
        normalized_he = np.zeros_like(normalized_he)

        #plt.imsave(f"/content/Croped_Images/{id}.png", normalized_he)
        plt.imsave(os.path.join(path, f"{id}.png"), normalized_he)


  0%|          | 27/63173 [00:06<3:50:47,  4.56it/s]

id: aabgacjd-1 <class 'numpy.ndarray'> center:  POINT (21180.29711021834 23570.801756719095)


  1%|          | 639/63173 [02:16<2:27:56,  7.05it/s]

id: achfhmpb-1 <class 'numpy.ndarray'> center:  POINT (1819.2338217144336 3020.403222477282)


  1%|          | 755/63173 [02:40<2:49:03,  6.15it/s]

...

 96%|█████████▋| 60949/63173 [3:14:42<06:50,  5.42it/s]

id: oamhndfi-1 <class 'numpy.ndarray'> center:  POINT (16316.499047573725 1110.629961655679)


 99%|█████████▉| 62447/63173 [3:19:31<02:07,  5.68it/s]

id: ogcccehg-1 <class 'numpy.ndarray'> center:  POINT (1709.444585742994 12368.17998520041)


100%|██████████| 63173/63173 [3:22:05<00:00,  5.21it/s]
